# Phase 04 — Frozen ChemBERTa embeddings and Gradient Boosting

This notebook evaluates ChemBERTa representations on the **same cleaned cohorts, frozen outer/inner folds, and seed** as Phase 03. The encoder is **frozen**; only its pooled embeddings are used to train Gradient Boosting. Two SMILES formats are tested: `raw` and `clean` (removing `[*]`).

Two representations are supported: **full** embeddings and **PCA 95%**, with PCA fitted inside each inner training fold to avoid leakage. Unlike descriptor GBR, the historical ChemBERTa baseline does **not** apply MI/permutation feature selection. The GBR tuning grid matches Phase 03, and outputs follow the same RunLogger/OOF convention.

Neither MongoDB data, Phase-02 manifests, nor Phase-03 functions are modified.

## 1. Configuration

Choose a cohort, seed, and the representations to evaluate. The frozen manifest must already exist for the selected seed. Reuse existing embedding CSVs when available; generate only missing files.

In [1]:
import hashlib
import json
import os
import sys
import tempfile
from datetime import datetime
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from pymongo import MongoClient

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'runlog.py').exists() and (p / 'src/final_comparison/datasets.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Run the notebook from the polymer repository')
sys.path.insert(0, str(ROOT))

from src.final_comparison.datasets import COLLECTIONS, DB_NAME, MD_COLUMNS, load_cohort
from src.final_comparison.splits import load_manifest, fold_ids
from src.final_comparison.gbr import PARAM_GRID, validate_frozen_splits
from src.final_comparison.chemberta import (
    MODEL_NAME, REPRESENTATIONS as AVAILABLE_REPRESENTATIONS,
    embedding_path, extract_embeddings, load_embeddings, benchmark_chemberta,
)
from runlog import RunLogger

load_dotenv(ROOT / '.env')

COHORT = 'Tg_md_common'  # Tg_md_common, Tg_bicerano, Tg_polyinfo
SEED = 28              # Same seed selects both manifest and GBR randomness
VARIANTS = ['raw']      # Add 'clean' for the preprocessing ablation
REPRESENTATIONS = ['full', 'pca95']
EXTRACT_MISSING_EMBEDDINGS = True
RUN_FULL_BENCHMARK = True
BATCH_SIZE = 32

COHORTS = {
    'Tg_md_common': ('bicerano', 'Tg', True),
    'Tg_bicerano': ('bicerano', 'Tg', False),
    'Tg_polyinfo': ('polyinfo', 'Tg', False),
}
dataset, target, require_md_coverage = COHORTS[COHORT]
assert VARIANTS and set(VARIANTS) <= {'raw', 'clean'}
assert REPRESENTATIONS and set(REPRESENTATIONS) <= set(AVAILABLE_REPRESENTATIONS)

MD_DIR = ROOT / 'data/md_features'
EMBEDDINGS_DIR = ROOT / 'data/embeddings'
SPLIT_DIR = ROOT / 'data/splits/v1'
MANIFESTS = [
    path for path in sorted(SPLIT_DIR.glob('*.json'))
    if (metadata := json.loads(path.read_text(encoding='utf-8'))).get('seed') == SEED
    and metadata.get('target') == target
    and metadata.get('cohort') == COHORT.removeprefix(f'{target}_')
]
if len(MANIFESTS) != 1:
    raise ValueError(f'Expected one frozen manifest for {COHORT}, seed {SEED}; found {len(MANIFESTS)}')
MANIFEST = MANIFESTS[0]
MANIFEST_SHA256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
print(f'{COHORT} | seed={SEED} | manifest={MANIFEST.relative_to(ROOT)}')
print(f'Variants: {VARIANTS} | representations: {REPRESENTATIONS}')

Tg_md_common | seed=28 | manifest=data\splits\v1\Tg_md_common.json
Variants: ['raw'] | representations: ['full', 'pca95']


## 2. Load and verify the frozen cohort

Load the exact cleaned target cohort through the Phase-01 function. `Tg_md_common` additionally requires complete MD coverage **for sample selection only**; MD features are never inputs to ChemBERTa. Verify every saved outer and inner split before continuing.

Embedding extraction uses the full cleaned parent collection (312 Bicerano samples, if available), so that the same cached embeddings can later serve both the full Bicerano and MD-common cohorts. The two cohorts remain distinct for evaluation.

In [2]:
md_sources = ({name: MD_DIR / name for name in MD_COLUMNS}
              if require_md_coverage else None)
if not os.getenv('MONGO_URI'):
    raise EnvironmentError('MONGO_URI must be set in .env')
with MongoClient(os.environ['MONGO_URI']) as mongo:
    frame, exclusions = load_cohort(
        dataset, target=target, md_dir=MD_DIR if require_md_coverage else None,
        client=mongo,
    )
    source_frame, _ = load_cohort(dataset, target=target, client=mongo)

manifest = load_manifest(MANIFEST, frame=frame, source_paths=md_sources)
ids, verified_folds = validate_frozen_splits(frame, manifest, fold_ids=fold_ids)
assert len(verified_folds) == manifest['n_outer_folds'] == 5
assert set(ids) <= set(source_frame.sample_id.astype(str))
print(f'Frozen cohort: {len(ids)} samples; {len(exclusions)} exclusions')
print(f'Embedding source cohort: {len(source_frame)} samples')

Frozen cohort: 135 samples; 180 exclusions
Embedding source cohort: 312 samples


## 3. Extract or reuse frozen embeddings

Existing files named `chemberta_<MongoCollection>_<variant>.csv` are reused (including historical exports), provided they cover **all** frozen IDs and have finite, uniquely indexed values. New extractions save a metadata sidecar recording encoder revision, pooling, source-ID/SMILES fingerprint, and CSV hash. Historical CSVs without this sidecar trigger a provenance warning and should be verified against the original encoder settings.

Embedding extraction is label-free. PCA is **not** performed here.

In [3]:
embeddings = {}
embedding_hashes = {}
for variant in VARIANTS:
    path = embedding_path(EMBEDDINGS_DIR, COLLECTIONS[dataset], variant)
    if not path.exists():
        if not EXTRACT_MISSING_EMBEDDINGS:
            raise FileNotFoundError(f'{path} does not exist; enable extraction')
        print(f'Extracting {variant} ChemBERTa embeddings for {len(source_frame)} polymers...')
        extract_embeddings(source_frame, path, variant=variant, model_name=MODEL_NAME,
                           batch_size=BATCH_SIZE)
    X = load_embeddings(path, ids, source_frame=source_frame,
                        variant=variant, model_name=MODEL_NAME)
    assert len(X) == len(frame) and X.index.tolist() == ids
    assert X.shape[1] == 768, f'Unexpected ChemBERTa hidden size: {X.shape[1]}'
    embeddings[variant] = X
    embedding_hashes[variant] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f'{variant}: {X.shape[0]} samples × {X.shape[1]} features; {path.name}')

assert all(X.index.equals(embeddings[VARIANTS[0]].index) for X in embeddings.values())
print(f'Frozen folds validated: {len(verified_folds)} outer × 5 inner')
print(f'Training enabled: {RUN_FULL_BENCHMARK}')

raw: 135 samples × 768 features; chemberta_biceranoPolymers_raw.csv
Frozen folds validated: 5 outer × 5 inner
Training enabled: True


## 4. Frozen nested cross-validation

Each `(variant, representation)` combination is an independent, logged run. The pipeline uses the same 54-candidate GBR grid as Phase 03, with the *stored* inner folds for tuning. For `pca95`, PCA is inside the pipeline, so no outer or inner validation observations contribute to fitted components. Hyperparameters are selected only on inner training/validation folds; the outer tests are used once for OOF predictions.

Outputs: `run.json`, `tables/oof_predictions.csv`, `tables/nested_cv_folds.csv`, and OOF-level MAE, R² and MAE/σ. Embedding extraction time is distinct from GBR training time and should be included in the Phase-06 cost discussion.

In [4]:
if RUN_FULL_BENCHMARK:
    summaries = []
    for variant, X in embeddings.items():
        for representation in REPRESENTATIONS:
            print(f'\nRunning {COHORT}: ChemBERTa {variant}/{representation}')
            path = embedding_path(EMBEDDINGS_DIR, COLLECTIONS[dataset], variant)
            metadata_path = path.with_suffix('.meta.json')
            metadata = (json.loads(metadata_path.read_text(encoding='utf-8'))
                        if metadata_path.is_file() else {'provenance': 'legacy_unverified'})
            config = {
                'db': DB_NAME, 'collection': COLLECTIONS[dataset], 'target': target,
                'cohort': COHORT, 'model_family': 'ChemBERTa_frozen_GBR',
                'encoder': MODEL_NAME, 'encoder_revision': metadata.get('model_revision'),
                'pooling': 'attention_mask_mean',
                'variant': variant, 'representation': representation,
                'embedding_source': str(path.relative_to(ROOT)),
                'embedding_sha256': embedding_hashes[variant],
                'embedding_extraction_seconds': metadata.get('extraction_seconds'),
                'embedding_provenance': 'verified' if metadata_path.exists() else 'legacy_unverified',
                'manifest': str(MANIFEST.relative_to(ROOT)), 'manifest_sha256': MANIFEST_SHA256,
                'random_state': SEED, 'use_md': False, 'use_mordred': False,
                'target_unit': 'K',
                'param_grid': {'regressor__' + k.removeprefix('gbr__'): v for k, v in PARAM_GRID.items()},
                'selection': 'none; optional PCA fitted within training folds',
            }
            log = RunLogger(config=config,
                            tag=f'phase04_{COHORT}_{variant}_{representation}_{datetime.now():%H%M%S_%f}')
            log.dataset(db=DB_NAME, collection=COLLECTIONS[dataset], n_excluded=len(exclusions))
            if not metadata_path.exists():
                log.note('Legacy embedding cache has no model/SMILES metadata; verify encoder and pooling provenance.')
            with tempfile.TemporaryDirectory(prefix='chemberta_cv_cache_') as cache:
                oof, folds, metrics = benchmark_chemberta(
                    frame, manifest, X, representation=representation,
                    fold_ids=fold_ids, log=log, param_grid=None,
                    seed=SEED, cache_dir=cache,
                )
            assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == MANIFEST_SHA256
            assert hashlib.sha256(path.read_bytes()).hexdigest() == embedding_hashes[variant]
            log.finish()
            summaries.append({'variant': variant, 'representation': representation, **metrics})
            display(pd.DataFrame([metrics]), folds, oof.head())
    display(pd.DataFrame(summaries))
else:
    print('Training is disabled; manifest, cohort and embeddings have been validated.')


Running Tg_md_common: ChemBERTa raw/full
[runlog] C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-24_1455_biceranoPolymers_nomd_phase04_Tg_md_common_raw_full_145525_133475
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
# Run `11-24_1455_biceranoPolymers_nomd_phase04_Tg_md_common_raw_full_145525_133475`

- **inizio**: 2026-11-24T14:55:25  |  **fine**: 2026-11-24T15:10:08  |  **durata**: 14.73 min
- **commit**: `36345b2`
- **tag**: phase04_Tg_md_common_raw_full_145525_133475

## Dati

- **sorgente**: MongoDB `PolymerPrediction.biceranoPolymers`
- **target**: Tg
- **polimeri usati**: 135
- **descrittori del monomero**: RDKit + Morgan + MACCS (`USE_MORDRED = False`)
- **range Tg**: 188 - 493 K (media 305, std 75

C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runlog.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(index_path), df], ignore_index=True)


,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,fit_seconds,n_components_mean
0,135,5,5,28.082661,0.752108,0.374821,28.082661,3.070898,883.262146,768.0


,fold,n_train,n_test,n_components,inner_MAE,MAE,R2,MAE_over_sigma,fit_seconds,best_params
0,1,108,27,768,28.400965,30.013855,0.723275,0.400597,171.216303,"{'regressor__learning_rate': 0.1, 'regressor__..."
1,2,108,27,768,30.465556,30.799439,0.658212,0.411082,178.008861,"{'regressor__learning_rate': 0.1, 'regressor__..."
2,3,108,27,768,29.667196,29.490832,0.723166,0.393616,176.968606,"{'regressor__learning_rate': 0.1, 'regressor__..."
3,4,108,27,768,32.871447,23.281189,0.842707,0.310736,186.195004,"{'regressor__learning_rate': 0.1, 'regressor__..."
4,5,108,27,768,27.753146,26.827990,0.768104,0.358075,170.873372,"{'regressor__learning_rate': 0.05, 'regressor_..."


,sample_id,fold,y_true,y_pred
0,698ddceeb7014ee539b3abeb,2,188.0,255.511337
1,698ddceeb7014ee539b3abec,5,190.0,215.191819
2,698ddceeb7014ee539b3abef,4,194.0,225.439064
3,698ddceeb7014ee539b3abf0,5,194.0,219.625040
4,698ddceeb7014ee539b3abf1,4,195.0,193.736837



Running Tg_md_common: ChemBERTa raw/pca95
[runlog] C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-24_1510_biceranoPolymers_nomd_phase04_Tg_md_common_raw_pca95_151009_155750
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits
# Run `11-24_1510_biceranoPolymers_nomd_phase04_Tg_md_common_raw_pca95_151009_155750`

- **inizio**: 2026-11-24T15:10:09  |  **fine**: 2026-11-24T15:11:34  |  **durata**: 1.42 min
- **commit**: `36345b2`
- **tag**: phase04_Tg_md_common_raw_pca95_151009_155750

## Dati

- **sorgente**: MongoDB `PolymerPrediction.biceranoPolymers`
- **target**: Tg
- **polimeri usati**: 135
- **descrittori del monomero**: RDKit + Morgan + MACCS (`USE_MORDRED = False`)
- **range Tg**: 188 - 493 K (media 305, std

C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runlog.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(index_path), df], ignore_index=True)


,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,fit_seconds,n_components_mean
0,135,5,5,31.337658,0.680414,0.418266,31.337658,9.967582,84.763215,21.6


,fold,n_train,n_test,n_components,inner_MAE,MAE,R2,MAE_over_sigma,fit_seconds,best_params
0,1,108,27,22,30.943713,32.988577,0.694027,0.440301,17.423401,"{'regressor__learning_rate': 0.1, 'regressor__..."
1,2,108,27,21,30.071285,36.601525,0.490514,0.488523,16.779053,"{'regressor__learning_rate': 0.1, 'regressor__..."
2,3,108,27,22,31.455833,20.906371,0.793905,0.279039,17.123096,"{'regressor__learning_rate': 0.05, 'regressor_..."
3,4,108,27,22,32.998821,21.871332,0.850067,0.291918,16.953887,"{'regressor__learning_rate': 0.05, 'regressor_..."
4,5,108,27,21,29.298405,44.320486,0.506792,0.591548,16.483779,"{'regressor__learning_rate': 0.1, 'regressor__..."


,sample_id,fold,y_true,y_pred
0,698ddceeb7014ee539b3abeb,2,188.0,226.090496
1,698ddceeb7014ee539b3abec,5,190.0,225.853411
2,698ddceeb7014ee539b3abef,4,194.0,202.429134
3,698ddceeb7014ee539b3abf0,5,194.0,234.603541
4,698ddceeb7014ee539b3abf1,4,195.0,208.038419


,variant,representation,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,fit_seconds,n_components_mean
0,raw,full,135,5,5,28.082661,0.752108,0.374821,28.082661,3.070898,883.262146,768.0
1,raw,pca95,135,5,5,31.337658,0.680414,0.418266,31.337658,9.967582,84.763215,21.6
